# 02 - Prototype DSPy Translation Pipeline

This notebook sets up and tests the DSPy-based translation pipeline:
1. Configure the language model (OpenAI or Ollama)
2. Test individual signatures
3. Test the full `TranslationChatbot` module
4. Evaluate on sample queries

**Prerequisites:** Run `01_parse_data.ipynb` first to create the dictionary and ChromaDB index.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

print(f'Project root: {project_root}')

## Step 1: Configure the Language Model

**Option A:** Groq (fast, free tier — requires API key in `.env`)  
**Option B:** Ollama (free, local — requires [Ollama](https://ollama.com) installed and `ollama pull llama3.2`)

In [ ]:
import dspy
from src.optimize import setup_lm

# --- CHOOSE YOUR LM PROVIDER ---
# Option A: Groq (recommended - fast & free tier)
api_key = os.getenv('GROQ_API_KEY')
if api_key and api_key != 'gsk_your-key-here':
    lm = setup_lm(provider='groq', model='llama-3.3-70b-versatile', api_key=api_key)
    print('Using Groq llama-3.3-70b-versatile')
else:
    # Option B: Ollama (local, free)
    lm = setup_lm(provider='ollama', model='llama3.2')
    print('Using Ollama llama3.2 (local)')

## Step 2: Test Individual Signatures

In [ ]:
from src.signatures import ClassifyQuery, TranslateWord, TranslatePhrase
from src.retriever import retrieve, get_collection

chroma_dir = str(project_root / 'chroma_db')
collection = get_collection(chroma_dir)

# Test query classification
classifier = dspy.Predict(ClassifyQuery)

test_queries = [
    "Translate 'love'",
    "How do you say 'good morning' in Akeanon?",
    "Translate: The house is beautiful",
    "What is the difference between Hiligaynon and Akeanon grammar?",
]

for q in test_queries:
    result = classifier(user_query=q)
    print(f'Query: "{q}"')
    print(f'  Type: {result.query_type} | Extracted: "{result.extracted_text}"')
    print()

In [ ]:
# Test word translation with dictionary context
word_translator = dspy.ChainOfThought(TranslateWord)

word = 'love'
context = retrieve(word, collection, chroma_dir, top_k=5)
print(f'Context for "{word}":')
for c in context:
    print(f'  {c}')

result = word_translator(
    english_word=word,
    part_of_speech='noun',
    dictionary_context=context,
)
print(f'\nHiligaynon: {result.hiligaynon}')
print(f'Akeanon: {result.akeanon}')
print(f'Notes: {result.notes}')
print(f'Reasoning: {result.reasoning}')

In [ ]:
# Test phrase translation
phrase_translator = dspy.ChainOfThought(TranslatePhrase)

phrase = 'good morning'
from src.retriever import retrieve_for_sentence
context = retrieve_for_sentence(phrase, collection, chroma_dir)
print(f'Context for "{phrase}":')
for c in context:
    print(f'  {c}')

result = phrase_translator(
    english_text=phrase,
    dictionary_context=context,
)
print(f'\nHiligaynon: {result.hiligaynon}')
print(f'Akeanon: {result.akeanon}')
print(f'Breakdown: {result.literal_breakdown}')
print(f'Notes: {result.notes}')

## Step 3: Test the Full TranslationChatbot Module

In [ ]:
from src.modules import TranslationChatbot, format_response

chatbot = TranslationChatbot(chroma_dir=chroma_dir)

# Test with various query types
test_messages = [
    "Translate 'water'",
    "How do you say 'I love you' in Akeanon?",
    "What is 'friend' in Hiligaynon?",
    "Translate: The food is delicious",
]

for msg in test_messages:
    print(f'\n{"="*60}')
    print(f'User: {msg}')
    print(f'{"="*60}')
    prediction = chatbot(user_query=msg)
    response = format_response(prediction)
    print(response)

## Step 4: Inspect LM Call History

In [ ]:
# See what prompts were sent to the LM
history = dspy.inspect_history(n=3)
print(history)

---
**Next:** Go to `03_optimize.ipynb` to optimize the pipeline with DSPy.